In [ ]:

import torch

from dataset_loaders import build_data_loaders
from utils.checkpoints import load_ae_from_path, load_cspn_from_path, load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import (
    plot_latent_space,
    show,
)


In [ ]:
ae_path = load_from_wandb("autoencoder_mnist", "best")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))


In [ ]:
dataset_cfg = DatasetConfig(
    name="mnist",
    channels=3,
    height=28,
    width=28,
    num_classes=10,
)

_, dataloader = build_data_loaders(dataset_cfg, batch_size=64)


In [ ]:
cspn_psi_path = load_from_wandb("cspn_mnist_psinet")
cspn_psi = load_cspn_from_path(cspn_psi_path, device=torch.device("mps"))

In [ ]:
label = 1
sample_labels = torch.tensor([label] * 3)

with torch.no_grad():
    samples_psi = cspn_psi.sample(sample_labels)
    sampled_images_psi_logits = ae.decode(samples_psi)
    sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)

show(sampled_images_psi, f"Samples from PSINet CSPN with label {label}")

In [ ]:
all_labels = torch.arange(10).repeat_interleave(5)

with torch.no_grad():
    samples_psi = cspn_psi.sample(all_labels)
    sampled_images_psi_logits = ae.decode(samples_psi)
    sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)

show(sampled_images_psi, "Samples from PSINet CSPN for all labels", width=5)

In [ ]:
labels = torch.arange(10).repeat_interleave(100)
with torch.no_grad():
    samples_psi = cspn_psi.sample(labels)

plot_latent_space(samples_psi, labels, title="Latent Space of PSINet CSPN on MNIST")

In [ ]:
random_latent = torch.randn(1, 8)
# tensor([0.3337, 0.3382, -0.2838, 0.4517, -0.3932, 0.5052, 0.3148, -0.7178]) is working
working_latent = torch.tensor([[0.3866, 0.4910, 0.5709, -0.4308, 0.5061, 0.5677, -0.3867, 0.5170]ff])
latents = torch.cat([working_latent, random_latent], dim=0)
with torch.no_grad():
    sampled_images_psi_logits = ae.decode(latents)
    sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)
print("Random latent samples shape:", random_latent.shape)
show(sampled_images_psi, "Samples from random latent vectors", width=10)